In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
import dlt



In [0]:
@dlt.view()
@dlt.expect_or_drop("product_id_not_null", "product_id IS NOT NULL")
def Dim_Products_view():
    df= spark.readStream.table("databricks_cat.silver.products")
    df = df.withColumn("DimProductKey",crc32("product_id"))
    #df = df.withColumn("isCurrent", when(col("__END_AT").isNotNull()|col('__END_AT')>=current_timestamp(),True).otherwise(False))
    return df

In [0]:
dlt.create_streaming_table("products")

In [0]:
from pyspark import pipelines as dp

dp.create_auto_cdc_flow(
  target = "products",
  source = "Dim_Products_view",
  keys = ["product_id"],
  sequence_by = "updated_timestamp",
  ignore_null_updates = True,
  apply_as_deletes = None,
  apply_as_truncates = None,
  column_list = None,
  except_column_list = None,
  stored_as_scd_type = 2,
  track_history_column_list = None,
  track_history_except_column_list = None,
  name = None,
  once = True
)